# Midtraining Dataset Extraction: Underrepresented & Rich Topics
This notebook focuses on extracting high-quality, pre-1930s documents from the `jbduran/think-dataset` based on Library of Congress (LC) classifications, targeting underrepresented subjects like Science and Technology.

> **Note on the source dataset:** we use `jbduran/think-dataset` (not `jbduran/think-dataset-clean-1930s`) for Stage 1. The `-clean-1930s` repo re-packs its shards after dropping ~3,714 anachronistic docs and ships **no** `audit_metadata.jsonl`, so its per-document metadata can't be positionally aligned to its text. `jbduran/think-dataset` is self-consistent (its audit metadata lines up row-for-row with its own shards). Revisit once `-clean-1930s` publishes regenerated audit metadata.

In [ ]:
import pandas as pd
from datasets import load_dataset
import json
import os
from google.colab import userdata

# Load a small streaming sample to inspect the shard structure.
# NOTE: the corpus parquet shards live at the REPO ROOT (shard_00000.parquet ...),
# not under data/, and the repo also has audit_metadata.jsonl / manifest.json at
# root. Select data_files='shard_*.parquet' so the loader streams only the corpus
# shards and doesn't try to fold the .jsonl metadata into the {'text': string} schema.
ds = load_dataset('jbduran/think-dataset', data_files='shard_*.parquet', split='train', streaming=True)
sample = next(iter(ds))

print("Sample Key(s):", sample.keys())
print("Sample Text Snippet:", sample['text'][:500] if 'text' in sample else "No text column found")

### 1. Metadata & Dataset Setup
We load the audit metadata to identify high-quality documents in Science, Technology, and Medicine.

In [ ]:
def check_metadata(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            # Try to read first line to see structure
            line = f.readline()
            return json.loads(line)
    return "File not found. Please upload or provide path."

# Placeholder for metadata inspection
# print(check_metadata('audit_metadata.jsonl'))

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = 'jbduran/think-dataset'

try:
    # Attempting to download the metadata files mentioned
    audit_path = hf_hub_download(repo_id=REPO_ID, filename='audit_metadata.jsonl', repo_type='dataset')
    manifest_path = hf_hub_download(repo_id=REPO_ID, filename='manifest.json', repo_type='dataset')

    print(f"Audit metadata downloaded to: {audit_path}")
    print(f"Manifest downloaded to: {manifest_path}")

    # Peek at the structure of the audit metadata
    with open(audit_path, 'r') as f:
        first_audit_record = json.loads(f.readline())
        print("\nFirst Audit Record Structure:")
        print(json.dumps(first_audit_record, indent=2))

    # Peek at manifest for topic distribution mapping
    with open(manifest_path, 'r') as f:
        manifest_data = json.load(f)
        print("\nManifest Keys:", manifest_data.keys())

except Exception as e:
    print(f"Error downloading metadata: {e}")

In [ ]:
import pandas as pd

# Load all audit records to see the subject landscape
audit_records = []
with open(audit_path, 'r') as f:
    for line in f:
        audit_records.append(json.loads(line))

df_audit = pd.DataFrame(audit_records)

# Show subject distribution to identify 'underrepresented' classes
print("Subject Distribution in Dataset:")
display(df_audit['topic_or_subject_gen'].value_counts())

# Filter for target 'rich/science' topics
target_topics = ['SCIENCE', 'TECHNOLOGY', 'MEDICINE']
rich_docs = df_audit[df_audit['topic_or_subject_gen'].isin(target_topics)]

print(f"\nFound {len(rich_docs)} documents in target topics: {target_topics}")
display(rich_docs[['title_src', 'topic_or_subject_gen', 'output_shard_index']].head())

### High-Quality Subset Definition
We filter for documents with OCR scores >= 99%, high tokenizability, and primarily English content to create the 'Gold' training set.

In [ ]:
# Get length of the first sample text we loaded earlier
sample_text_len = len(sample['text'].split())
metadata_word_count = first_audit_record['word_count']

print(f"Shard Sample Word Count: {sample_text_len}")
print(f"Metadata Word Count: {metadata_word_count}")

if abs(sample_text_len - metadata_word_count) < (0.05 * metadata_word_count):
    print("\nRESULT: Shards appear to be 1-to-1 with documents. Extraction is a simple shard-level filter.")
else:
    print("\nRESULT: Shards may contain multiple documents. We need to find the delimiters.")

### Document Extraction Strategy
Since each row is a document, we can simply filter the `df_audit` for our target topics, then use the `output_shard_index` and the relative index within that shard to pull the specific text rows.

In [ ]:
# We will create a list of the target documents and their locations
target_docs = df_audit[df_audit['topic_or_subject_gen'].isin(['SCIENCE', 'TECHNOLOGY', 'MEDICINE'])]

# Grouping by shard will make extraction more efficient
shard_groups = target_docs.groupby('output_shard_filename')

print(f"We need to visit {len(shard_groups)} unique shards to collect {len(target_docs)} documents.")

# Let's define a function to pull a sample of these rich documents for verification
def get_rich_sample(n=5):
    subset = target_docs.sample(n)
    results = []
    for idx, row in subset.iterrows():
        results.append({
            'title': row['title_src'],
            'topic': row['topic_or_subject_gen'],
            'shard': row['output_shard_filename']
        })
    return pd.DataFrame(results)

display(get_rich_sample())

In [ ]:
import numpy as np

# Calculate precise stats for the target subset
target_stats = target_docs.agg({
    'char_count': 'sum',
    'word_count': 'sum',
    'token_count_o200k_base_gen': 'sum'
})

total_chars = target_stats['char_count']
total_gb = total_chars / (1024**3)
# Target 250M chars per shard
chars_per_shard = 250_000_000
est_num_shards = int(np.ceil(total_chars / chars_per_shard))

print(f"--- Subset Profile ---")
print(f"Target Documents: {len(target_docs):,}")
print(f"Total Characters: {total_chars:,} (~{total_gb:.2f} GB)")
print(f"Total Words:      {target_stats['word_count']:,}")
print(f"Estimated Shards: {est_num_shards} (at 250M chars/shard)")

### Quality-Based Pruning
We will now filter the 20k target documents by `ocr_score_gen` and `tokenizability_score` to reduce the volume from 3.4B tokens to a high-density 'gold' set.

In [ ]:
# Filtering for the 'Best of the Best'
high_quality_rich = target_docs[
    (target_docs['ocr_score_gen'] >= 99.0) &
    (target_docs['tokenizability_score'] >= 99.0) &
    (target_docs['english_proportion'] >= 0.98)
]

# Calculate new stats
hq_stats = high_quality_rich.agg({
    'char_count': 'sum',
    'token_count_o200k_base_gen': 'sum'
})

print(f"--- High Quality Subset (OCR >= 99) ---")
print(f"Documents: {len(high_quality_rich):,}")
print(f"Characters: {hq_stats['char_count']:,}")
print(f"Estimated Tokens: {hq_stats['token_count_o200k_base_gen']:,}")
print(f"Approximate MB (uncompressed text): {hq_stats['char_count'] / (1024**2):.2f} MB")

### Extraction & Sharding Algorithm
This algorithm streams the source shards and packs the filtered high-quality documents into new Parquet files optimized for training (~250M characters per shard).

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

# Grouping the HQ subset by their original location to optimize streaming
hq_shard_groups = high_quality_rich.groupby('output_shard_filename')

print(f"Optimized plan: We only need to stream {len(hq_shard_groups)} source shards.")

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import tqdm
import os

# Configuration
CHARS_PER_SHARD = 250_000_000
OUTPUT_DIR = 'midtraining_gold_shards_v1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save_parquet_shard(batch_data, shard_num):
    """Saves a list of dicts as a Parquet shard with full metadata preservation."""
    df_shard = pd.DataFrame(batch_data)
    table = pa.Table.from_pandas(df_shard)
    filename = os.path.join(OUTPUT_DIR, f"gold_sci_tech_shard_{shard_num:03d}.parquet")
    pq.write_table(table, filename)
    return filename

# Extraction Algorithm State
current_shard_rows = []
current_char_count = 0
shard_index = 0

# We process shards in numerical order (000 -> 472) to ensure non-random packing
sorted_shard_filenames = sorted(hq_shard_groups.groups.keys())

print(f"[Diagnostic] Algorithm initialized for {len(high_quality_rich)} documents.")

# --- EXTRACTION LOOP (DRY RUN LOGIC) ---
def run_extraction_algorithm(dry_run=True):
    global current_shard_rows, current_char_count, shard_index

    for shard_name in tqdm.tqdm(sorted_shard_filenames, desc="Processing Sequence"):
        # Target metadata for this specific shard
        target_metadata_in_shard = high_quality_rich[high_quality_rich['output_shard_filename'] == shard_name]
        # Global audit list for finding local row positions
        all_shard_metadata = df_audit[df_audit['output_shard_filename'] == shard_name]

        if not dry_run:
            # Load this specific Parquet file. The corpus shards live at the REPO
            # ROOT (shard_00000.parquet ...), not under data/, so data_files is
            # just the bare shard filename. This positional row lookup below is
            # only valid because df_audit and these shards come from the SAME repo
            # (jbduran/think-dataset) -- do NOT swap in a re-packed/cleaned repo
            # without a stable join key.
            source_shard = load_dataset(REPO_ID, data_files=shard_name, split='train')

        for global_idx, meta_row in target_metadata_in_shard.iterrows():
            local_idx = all_shard_metadata.index.get_loc(global_idx)

            if dry_run:
                # In dry run, we just simulate character accumulation
                doc_len = meta_row['char_count']
                current_char_count += doc_len
            else:
                doc_text = source_shard[int(local_idx)]['text']
                # Preserve full metadata for future dissection
                row = {
                    'text': doc_text,
                    'barcode': meta_row['barcode_src'],
                    'title': meta_row['title_src'],
                    'subject': meta_row['topic_or_subject_gen'],
                    'ocr_score': meta_row['ocr_score_gen'],
                    'year': meta_row['resolved_year'],
                    'orig_shard': shard_name,
                    'orig_row': local_idx
                }
                current_shard_rows.append(row)
                current_char_count += len(doc_text)

            # Check shard boundary
            if current_char_count >= CHARS_PER_SHARD:
                if not dry_run: save_parquet_shard(current_shard_rows, shard_index)
                shard_index += 1
                current_shard_rows = []
                current_char_count = 0

    print(f"\nAlgorithm complete. Total shards that will be generated: {shard_index + (1 if current_char_count > 0 else 0)}")

# Running diagnostic dry run
run_extraction_algorithm(dry_run=True)

## Stage 2: Sourcing Additional STEM Data (Internet Archive)
To reach our ~4B token goal, we will tap into the Internet Archive's (IA) vast collection of public domain pre-1930s literature. We prioritize collections known for high-quality scans like `biodiversity`, `smithsonian`, and `americana`.

In [ ]:
!pip install internetarchive langdetect

In [ ]:
from internetarchive import search_items, get_session
import pandas as pd

print("Attempting a standard search call with custom User-Agent...")

s = get_session(config={'allow_unlimited_results': True})
s.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) GoldSTEMBot/1.0'})

# NOTE: search_items() takes `archive_session`, not `session` -- that kwarg
# typo was the root cause of the earlier failure.
# Pre-1930 is a hard requirement, so the date range is baked directly into
# the query (defense #1). We double check each item's actual metadata date
# later (defense #2), since IA date/year fields are inconsistent.
query = 'collection:biodiversity AND mediatype:texts AND language:eng AND date:[1000-01-01 TO 1929-12-31]'

try:
    search = search_items(query, fields=['identifier', 'title', 'date', 'year'], archive_session=s)

    debug_results = []
    for i, res in enumerate(search):
        debug_results.append(res)
        if i >= 4:
            break

    if debug_results:
        debug_df = pd.DataFrame(debug_results)
        print("\nSuccess! API returned metadata:")
        display(debug_df)
    else:
        print("Search returned no results.")

except Exception as e:
    print(f"\nSearch call failed: {e}")

### Strategy for Quality at Scale
To ensure 'least artifacted' data, we will:
1.  **Filter by Collection**: Target collections like `biodiversity` (BHL) which often have superior OCR/Metadata.
2.  **Metadata Validation**: Check for `language:eng` and specific `format` tags (e.g., 'DjVuTXT' or 'Full Text').
3.  **Parallel Downloader**: Use IA's Python API to fetch only the `.txt` or `.json` metadata to estimate token counts before committing to a full download.

In [ ]:
from collections import OrderedDict

# Math is the top-priority subject; broader STEM collections/subjects are a
# supplementary pool. `biodiversity` and `medicalheritagelibrary` are
# inherently STEM-scoped collections; general `subject:` terms are needed
# to reach math/STEM content outside those two collections.
MATH_SUBJECT_TERMS = [
    'mathematics', 'algebra', 'geometry', 'calculus', 'arithmetic',
    'trigonometry', 'number theory', 'mathematical analysis', 'logic',
]
STEM_COLLECTIONS = ['biodiversity', 'medicalheritagelibrary']
STEM_SUBJECT_TERMS = [
    'science', 'chemistry', 'physics', 'biology', 'astronomy',
    'engineering', 'technology', 'geology', 'botany', 'zoology', 'medicine',
]
DATE_RANGE = 'date:[1000-01-01 TO 1929-12-31]'
SEARCH_FIELDS = ['identifier', 'title', 'date', 'year', 'subject', 'collection', 'creator']

def build_subject_query(terms):
    clause = ' OR '.join(f'subject:"{t}"' for t in terms)
    return f'mediatype:texts AND language:eng AND {DATE_RANGE} AND ({clause})'

def harvest_identifiers(query, source_tag, limit=50000):
    """Runs a search_items query, tags each hit with source_tag, and caps
    result volume so an overly broad query can't run indefinitely."""
    search = search_items(query, fields=SEARCH_FIELDS, archive_session=s)
    rows = []
    for i, res in enumerate(search):
        res = dict(res)
        res['source_tag'] = source_tag
        rows.append(res)
        if limit and i + 1 >= limit:
            break
    return rows

# Tier 1: Math -- top priority, no volume cap beyond the sanity limit above.
math_candidates = harvest_identifiers(build_subject_query(MATH_SUBJECT_TERMS), 'math')
print(f"Tier 1 (Math): {len(math_candidates)} candidates")

# Tier 2: dedicated STEM collections.
stem_candidates = []
for coll in STEM_COLLECTIONS:
    q = f'collection:{coll} AND mediatype:texts AND language:eng AND {DATE_RANGE}'
    rows = harvest_identifiers(q, coll)
    print(f"Tier 2 ({coll}): {len(rows)} candidates")
    stem_candidates.extend(rows)

# Tier 2b: broader STEM subject match outside the dedicated collections.
stem_subject_candidates = harvest_identifiers(build_subject_query(STEM_SUBJECT_TERMS), 'stem_subject')
print(f"Tier 2b (STEM subject match): {len(stem_subject_candidates)} candidates")
stem_candidates.extend(stem_subject_candidates)

# Combine + dedupe by identifier; math entries added last so they win on collision.
all_candidates = OrderedDict()
for row in stem_candidates + math_candidates:
    all_candidates[row['identifier']] = row

df_ia_candidates = pd.DataFrame(all_candidates.values())
print(f"\nTotal unique IA candidates: {len(df_ia_candidates)}")
display(df_ia_candidates['source_tag'].value_counts())

In [ ]:
import re
import time
import random
import requests
import langdetect
from langdetect import DetectorFactory
import nltk

DetectorFactory.seed = 0
nltk.download('words', quiet=True)
from nltk.corpus import words as nltk_words
ENGLISH_VOCAB = set(w.lower() for w in nltk_words.words())

# Quality bar: heuristic pre-filter (length/alpha-ratio/language) PLUS a
# dictionary-hit-ratio OCR-confidence proxy, since IA has no precomputed
# ocr_score_gen like the HF dataset did. Calibrated against a known-garbled
# scan (front-matter noise, ~0.87) and a clean scan (~0.85) -- both legible
# documents score in the 0.8+ range, so we gate at 0.80.
MIN_CHARS = 2000
# IA djvu.txt is raw OCR of scanned pages -- newlines, page numbers, and OCR
# spacing mean a perfectly clean book only reaches ~0.73-0.75 alpha ratio
# (measured on advancementof00bacouoft=0.735, 1000hintsonflowe00croyuoft=0.754,
# 100henhowtogethe00wolv=0.748). The old 0.85 gate was miscalibrated and
# rejected ~36K legitimate docs before this was fixed. 0.65 clears real scans
# with margin while still catching number-table / image-noise garbage; the
# OCR-confidence dict-hit gate below (>=0.80) is the actual quality filter.
MIN_ALPHA_RATIO = 0.65
MIN_OCR_CONFIDENCE = 0.80

def ocr_confidence(text, sample_chars=50000):
    sample = text[:sample_chars]
    tokens = re.findall(r"[A-Za-z]+", sample)
    if len(tokens) < 20:
        return 0.0
    hits = sum(1 for t in tokens if t.lower() in ENGLISH_VOCAB)
    return hits / len(tokens)

def alpha_ratio(text):
    if not text:
        return 0.0
    return sum(1 for c in text if c.isalpha()) / len(text)

def passes_quality_bar(text):
    if len(text) < MIN_CHARS:
        return False, 'too_short'
    if alpha_ratio(text) < MIN_ALPHA_RATIO:
        return False, 'low_alpha_ratio'
    try:
        if langdetect.detect(text[:5000]) != 'en':
            return False, 'not_english'
    except langdetect.lang_detect_exception.LangDetectException:
        return False, 'lang_detect_failed'
    conf = ocr_confidence(text)
    if conf < MIN_OCR_CONFIDENCE:
        return False, 'low_ocr_confidence'
    return True, conf

# --- IA fetch with retries -------------------------------------------------
# IA regularly serves individual items off flaky/slow download nodes
# (e.g. dnNNNNNN.eu.archive.org), and a single ConnectTimeout used to
# propagate out of the worker, surface at future.result(), and kill the ENTIRE
# ~95K-item run. Over that many requests, drawing at least one dead node is
# effectively guaranteed -- so the fetch must be self-healing:
#   * shorter per-attempt connect timeout so a dead node fails fast instead of
#     blocking a worker for a full minute (read timeout stays generous -- some
#     djvu.txt files are large and legitimately slow to stream);
#   * bounded retries with exponential backoff + jitter, which also re-resolves
#     archive.org and usually lands on a healthy node the next time;
#   * ALL network exceptions (timeouts, connection resets, chunked-read errors)
#     are caught. On total failure we return None, and the caller logs the item
#     as a skippable 'rejected_no_text' -- one bad item costs one item, not
#     the whole run.
# 404 / non-200 (item genuinely has no _djvu.txt) is NOT retried -- it's a
# permanent condition, so we return None immediately.
FETCH_MAX_RETRIES = 4
FETCH_CONNECT_TIMEOUT = 15   # seconds to establish the connection (fail fast on a dead node)
FETCH_READ_TIMEOUT = 120     # seconds to stream the body (djvu.txt can be big)

def fetch_ia_text(identifier, max_retries=FETCH_MAX_RETRIES):
    url = f'https://archive.org/download/{identifier}/{identifier}_djvu.txt'
    for attempt in range(max_retries):
        try:
            r = requests.get(
                url,
                headers={'User-Agent': 'Mozilla/5.0 GoldSTEMBot/1.0'},
                timeout=(FETCH_CONNECT_TIMEOUT, FETCH_READ_TIMEOUT),
            )
            if r.status_code == 200:
                return r.text
            # 404 etc. -> item has no djvu.txt; permanent, don't retry.
            if r.status_code < 500:
                return None
            # 5xx -> IA-side transient; fall through to backoff+retry.
        except requests.RequestException:
            # ConnectTimeout / ReadTimeout / ConnectionError / ChunkedEncodingError etc.
            pass
        # Backoff before the next attempt (skip the wait after the final one).
        if attempt < max_retries - 1:
            time.sleep((2 ** attempt) + random.random())
    # Exhausted retries -- signal a fetch failure to the caller (it will be
    # logged as a skippable status, not raised).
    return None

def resolve_year(row):
    """Defense #2: re-validate pre-1930 from actual item metadata, independent
    of the query-level date filter (IA date/year fields are inconsistent)."""
    year = row.get('year')
    if isinstance(year, int):
        return year
    date_str = row.get('date', '')
    m = re.match(r'(\d{4})', str(date_str))
    return int(m.group(1)) if m else None

In [ ]:
import re
import tqdm

def normalize_title(t):
    if not isinstance(t, str):
        return ''
    t = t.lower()
    t = re.sub(r'[^a-z0-9\s]', '', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

# NOTE: Cross-stage dedup against Stage 1 is intentionally deferred.
# Stage 1's source dataset (jbduran/think-dataset) is currently being
# re-stripped by a teammate to remove anachronistic footers / bad-quality
# tokens, so Stage 1 isn't finalized in this session (df_audit may not even
# be loaded). The combined dedup + "gold-digging" pass across both stages
# happens later, once that reprocessing is done.
#
# For now we only dedupe *within* the IA harvest itself (different
# scans/editions of the same work), and we do it on the candidate list
# BEFORE downloading -- cheaper, and it means every download in the next
# cell is guaranteed non-duplicate.
#
# --- WHY THIS WAS REWRITTEN ---
# The previous version blocked titles by (first word, length-decile) and ran an
# O(bucket-size) difflib.SequenceMatcher scan per item. Old book titles
# overwhelmingly start with "the"/"a"/"an", so those buckets still held
# thousands of titles even after the length split; combined with difflib being
# O(len**2) per comparison, the pass was effectively quadratic and never
# finished on ~146K candidates (its own comment admitted "minutes and still not
# done") -- that hung cell is the bug being fixed here.
#
# Replacement: a hash-based grouping on a normalized *dedup key*, which is O(n)
# with a plain dict. The key lower-cases, strips punctuation, and also removes a
# trailing edition / volume / year marker -- which is exactly how IA duplicate
# editions of the same work differ ("... vol 1", "... 2nd ed", "... 1899").
# This collapses the real duplicate editions (empirically the vast majority)
# in a fraction of a second instead of hanging. It deliberately does NOT try to
# catch fuzzy mid-string differences (e.g. a single OCR typo inside the title);
# that long tail wasn't worth an unbounded-runtime fuzzy pass, and a proper
# content-level near-dup pass happens later in the deferred cross-stage stage.
df_ia_candidates['title_norm'] = df_ia_candidates['title'].apply(normalize_title)

# Trailing edition/volume/year markers that distinguish scans of the SAME work.
# Applied repeatedly so stacked markers ("... vol 1 1899") all peel off.
_EDITION_SUFFIX_RE = re.compile(
    r'\s+('
    r'vol(ume)?\s*\d+|v\s*\d+|pt\s*\d+|part\s*\d+|no\s*\d+|'
    r'\d+(st|nd|rd|th)?\s+ed(ition)?|new\s+ed(ition)?|revised(\s+ed(ition)?)?|'
    r'first\s+ed(ition)?|second\s+ed(ition)?|third\s+ed(ition)?|fourth\s+ed(ition)?|'
    r'\d{4}'
    r')$'
)

def dedup_key(norm):
    prev = None
    while prev != norm:
        prev = norm
        norm = _EDITION_SUFFIX_RE.sub('', norm).strip()
    return norm

# Track how many near-duplicates get collapsed into each survivor, for the
# `likely_duplicate_count` metadata field later.
dup_counts = {}

def find_duplicate_group_reps(identifiers, norms):
    """O(n) near-duplicate detection by exact match on a normalized dedup key.

    Iterates in input order so the first-seen identifier of a duplicate group is
    its representative (same convention as the original implementation). Every
    later identifier whose key already has a representative is folded into that
    rep and bumps its dup count. Empty/non-string titles are always their own
    rep (never merged with each other)."""
    rep_for_identifier = {}
    key_to_rep = {}
    for ident, norm in tqdm.tqdm(zip(identifiers, norms), total=len(identifiers), desc="Dedup candidates"):
        if not norm:
            rep_for_identifier[ident] = ident
            continue
        key = dedup_key(norm)
        rep = key_to_rep.get(key)
        if rep is None:
            key_to_rep[key] = ident
            rep_for_identifier[ident] = ident
        else:
            rep_for_identifier[ident] = rep
            dup_counts[rep] = dup_counts.get(rep, 0) + 1
    return rep_for_identifier

before_intra = len(df_ia_candidates)
rep_map = find_duplicate_group_reps(df_ia_candidates['identifier'], df_ia_candidates['title_norm'])
df_ia_candidates['is_dup_rep'] = df_ia_candidates['identifier'].map(lambda i: rep_map[i] == i)
df_ia_candidates_dedup = (
    df_ia_candidates[df_ia_candidates['is_dup_rep']]
    .drop(columns=['title_norm', 'is_dup_rep'])
    .reset_index(drop=True)
)
df_ia_candidates_dedup['likely_duplicate_count'] = df_ia_candidates_dedup['identifier'].map(lambda i: dup_counts.get(i, 0))

print(f"Intra-Stage-2 candidate dedup: {before_intra} -> {len(df_ia_candidates_dedup)} candidates "
      f"({before_intra - len(df_ia_candidates_dedup)} duplicate editions collapsed)")
display(df_ia_candidates_dedup['source_tag'].value_counts())

In [ ]:
import os
import json
import numpy as np
import tiktoken
from huggingface_hub import HfApi, hf_hub_download
from google.colab import userdata

# --- HF setup ---
# Token comes from Colab Secrets (HF_TOKEN_Z), never pasted into the notebook.
HF_TOKEN = userdata.get('HF_TOKEN_Z')
HF_REPO_ID = 'zachnorton03/think-midtrain'
STAGE2_PREFIX = 'stage2'
STAGE2_CHARS_PER_SHARD = 250_000_000  # same rough shard size as jbduran/think-dataset

api = HfApi(token=HF_TOKEN)
TIKTOKEN_ENC = tiktoken.get_encoding('o200k_base')

# --- Token budget for the COMBINED midtrain set (Stage 1 + Stage 2) ---
# Preferred target ~2.3B tokens (~1/10 of the base pretraining set), hard
# cap 4B tokens. Stage 1 already accounts for ~700M of that.
if 'hq_stats' in globals():
    STAGE1_TOKENS_ESTIMATE = int(hq_stats['token_count_o200k_base_gen'])
else:
    STAGE1_TOKENS_ESTIMATE = 700_000_000  # Stage 1 not run this session; user-provided estimate

TOTAL_TARGET_TOKENS = 2_300_000_000
TOTAL_HARD_CAP_TOKENS = 4_000_000_000
STAGE2_TARGET_TOKENS = max(0, TOTAL_TARGET_TOKENS - STAGE1_TOKENS_ESTIMATE)
STAGE2_HARD_CAP_TOKENS = max(0, TOTAL_HARD_CAP_TOKENS - STAGE1_TOKENS_ESTIMATE)

print(f"Stage 1 tokens (estimate): {STAGE1_TOKENS_ESTIMATE:,}")
print(f"Stage 2 preferred target:  {STAGE2_TARGET_TOKENS:,} tokens")
print(f"Stage 2 hard cap:          {STAGE2_HARD_CAP_TOKENS:,} tokens")

CHECKPOINT_PATH_IN_REPO = f'{STAGE2_PREFIX}/checkpoint.jsonl'
LOCAL_CHECKPOINT_PATH = '/tmp/stage2_checkpoint.jsonl'

# --- Resume support: pull any existing checkpoint AND audit metadata so a ---
# --- rerun after a disconnect continues cleanly (correct shard_index and ---
# --- running token total) instead of restarting or silently overwriting ---
# --- already-uploaded shards. ---
processed = {}
try:
    existing_ckpt = hf_hub_download(repo_id=HF_REPO_ID, filename=CHECKPOINT_PATH_IN_REPO, repo_type='dataset', token=HF_TOKEN)
    with open(existing_ckpt) as f:
        for line in f:
            rec = json.loads(line)
            processed[rec['identifier']] = rec['status']
    print(f"Resumed checkpoint: {len(processed)} identifiers already processed.")
except Exception:
    print("No existing checkpoint found on HF -- starting fresh.")

all_meta_records = []
try:
    existing_audit = hf_hub_download(repo_id=HF_REPO_ID, filename=f'{STAGE2_PREFIX}/audit_metadata.jsonl', repo_type='dataset', token=HF_TOKEN)
    with open(existing_audit) as f:
        for line in f:
            all_meta_records.append(json.loads(line))
    print(f"Resumed audit metadata: {len(all_meta_records)} records already uploaded.")
except Exception:
    print("No existing audit metadata found on HF -- starting fresh.")

running_token_total = sum(r.get('token_count_o200k_base_gen', 0) for r in all_meta_records)
resume_shard_index = max((r['output_shard_index'] for r in all_meta_records), default=-1) + 1
print(f"Resumed token total: {running_token_total:,} tokens already banked.")
print(f"Resuming at shard index: {resume_shard_index}")

_ckpt_file = open(LOCAL_CHECKPOINT_PATH, 'a')

def log_checkpoint(identifier, status):
    processed[identifier] = status
    _ckpt_file.write(json.dumps({'identifier': identifier, 'status': status}) + '\n')
    _ckpt_file.flush()

def push_checkpoint():
    api.upload_file(
        path_or_fileobj=LOCAL_CHECKPOINT_PATH, path_in_repo=CHECKPOINT_PATH_IN_REPO,
        repo_id=HF_REPO_ID, repo_type='dataset', commit_message='Update stage2 checkpoint',
    )

# --- Topic mapping: map our IA source_tag/subject onto jbduran's broad LC-style buckets ---
def classify_topic(row):
    tag = row.get('source_tag')
    if tag == 'math':
        return 'SCIENCE', 1.0  # LC class QA (Mathematics) falls under Q (Science)
    if tag == 'medicalheritagelibrary':
        return 'MEDICINE', 1.0
    if tag == 'biodiversity':
        return 'SCIENCE', 0.9
    subj = str(row.get('subject', '')).lower()
    if 'medic' in subj:
        return 'MEDICINE', 0.7
    if 'engineer' in subj or 'technolog' in subj:
        return 'TECHNOLOGY', 0.7
    return 'SCIENCE', 0.7

# --- Metadata schema matched to jbduran/think-dataset's audit_metadata.jsonl fields. ---
# Fields with no IA analog (ocr_score_src, ocr_disagreement, page_count_src)
# are left None, consistent with that dataset's own convention of nullable
# fields (date2_src was None in the sample record we inspected).
# A few additive provenance fields are appended beyond the original schema
# (ia_identifier/ia_url/stem_focus_tag/dataset_stage) since they don't
# conflict with it and are necessary for an IA-sourced document.
def build_metadata(row, text, resolved_year, ocr_conf):
    topic, topic_score = classify_topic(row)
    try:
        lang_probs = {l.lang: l.prob for l in langdetect.detect_langs(text[:5000])}
        eng_proportion = lang_probs.get('en', 0.0)
        lang_code = max(lang_probs, key=lang_probs.get) if lang_probs else 'unk'
    except Exception:
        eng_proportion, lang_code = 0.0, 'unk'

    creator = row.get('creator')
    if isinstance(creator, list):
        creator = '; '.join(creator)

    sentence_count = len(re.split(r'[.!?]+\s', text))
    token_count = len(TIKTOKEN_ENC.encode(text, disallowed_special=()))

    return {
        'author_src': creator,
        'barcode_src': row['identifier'],  # closest analog to a physical/scan ID
        'char_count': len(text),
        'date1_src': row.get('date'),
        'date2_src': None,
        'date_types_src': 'Internet Archive metadata date (single)',
        'english_proportion': eng_proportion,
        'is_val': bool(np.random.rand() < 0.03),
        'language_gen': 'eng' if lang_code == 'en' else lang_code,
        'likely_duplicate_count': int(row.get('likely_duplicate_count', 0)),
        'ocr_disagreement': None,          # no second independent OCR pass to compare against
        'ocr_score_gen': round(ocr_conf * 100, 2),
        'ocr_score_src': None,             # no separate "src" OCR pass exists for IA
        'page_count_src': None,
        'resolved_year': resolved_year,
        'sentence_count': sentence_count,
        'title_src': row.get('title'),
        'token_count_o200k_base_gen': token_count,
        # Approximation (jbduran's original tokenizability method is unknown to us) --
        # reuses the alpha-character ratio as a legibility-for-tokenization proxy.
        'tokenizability_score': round(min(1.0, alpha_ratio(text)) * 100, 2),
        'topic_or_subject_gen': topic,
        'topic_or_subject_score_gen': topic_score,
        'word_count': len(text.split()),
        # --- additive fields, not part of the original jbduran schema ---
        'ia_identifier': row['identifier'],
        'ia_url': f"https://archive.org/details/{row['identifier']}",
        'stem_focus_tag': row.get('source_tag'),
        'dataset_stage': 'stage_2',
    }

print(f"HF target: {HF_REPO_ID}/{STAGE2_PREFIX}/  (shard size ~{STAGE2_CHARS_PER_SHARD:,} chars)")

In [ ]:
# --- FRESH START: wipe the stage2/ outputs on HF -----------------------------
# The earlier run rejected ~37K legitimate docs because MIN_ALPHA_RATIO was
# miscalibrated (0.85 vs. the ~0.74 real IA djvu alpha ratio) and uploaded ZERO
# shards. Those identifiers are recorded as 'rejected_quality:low_alpha_ratio'
# in stage2/checkpoint.jsonl on HF. Since the resume logic in the cell above
# reloads that checkpoint and the main loop SKIPS anything already in
# `processed`, a plain re-run would silently skip all those now-fixable docs.
#
# Because the bad run produced no usable shards, the clean fix is to delete the
# whole stage2/ output tree on HF and start over from scratch under the
# corrected 0.65 gate.
#
# GUARD: set to True to actually delete. Leave False on normal resume runs so
# this cell is a no-op and can't wipe good data by accident.
CONFIRM_FRESH_START = True

if CONFIRM_FRESH_START:
    from huggingface_hub import list_repo_files
    existing = [
        f for f in list_repo_files(repo_id=HF_REPO_ID, repo_type='dataset', token=HF_TOKEN)
        if f.startswith(f'{STAGE2_PREFIX}/')
    ]
    print(f"Deleting {len(existing)} file(s) under {STAGE2_PREFIX}/ on {HF_REPO_ID} ...")
    for f in existing:
        api.delete_file(
            path_in_repo=f, repo_id=HF_REPO_ID, repo_type='dataset',
            commit_message=f'Fresh start: remove {f}',
        )
        print(f"  deleted {f}")

    # Also reset the in-memory state populated by the cell above, so if you don't
    # re-run that cell the run still starts genuinely fresh.
    processed = {}
    all_meta_records = []
    running_token_total = 0
    resume_shard_index = 0
    try:
        _ckpt_file.close()
    except Exception:
        pass
    _ckpt_file = open(LOCAL_CHECKPOINT_PATH, 'w')  # truncate the local checkpoint too
    print("\nFresh start complete: HF stage2/ cleared and in-memory state reset.")
    print(f"processed={len(processed)}, records={len(all_meta_records)}, "
          f"tokens={running_token_total}, next_shard_index={resume_shard_index}")
else:
    print("CONFIRM_FRESH_START is False -- skipping wipe (normal resume run).")

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import tqdm
import traceback
from concurrent.futures import ThreadPoolExecutor

# --- Throughput knobs -------------------------------------------------------
# The old loop downloaded each IA book serially (~6-7s/doc of pure network
# wait) AND downloaded every candidate before deciding to reject it on date.
# Two fixes here, both aimed at the network bottleneck (a Colab CPU is not the
# limiter -- archive.org round-trips are):
#   1. DATE PRE-FILTER: resolve_year() uses only the search metadata we already
#      have, so we drop >=1930 / unparseable candidates *before* any download.
#   2. PARALLEL FETCH: a thread pool runs fetch_ia_text + passes_quality_bar +
#      build_metadata concurrently across DOWNLOAD_WORKERS books. The main
#      thread stays single-threaded for all shard/token/checkpoint bookkeeping,
#      so the deterministic packing order and resume logic are unchanged --
#      results are consumed in submission (priority) order, not completion order.
DOWNLOAD_WORKERS = 16

shard_text_buffer = []
shard_meta_buffer = []
shard_char_count = 0
shard_index = resume_shard_index      # continues from the last uploaded shard, doesn't overwrite it
# all_meta_records / running_token_total are already initialized in the cell above (resumed from HF if present)
df_ia_gold_rows = []  # NOTE: only reflects docs downloaded *this run* -- on a resumed run this
                       # won't include prior-run docs' text (only their metadata persisted). Use the
                       # HF read-back verification cell below for a full-corpus view after a resume.

def flush_shard():
    global shard_text_buffer, shard_meta_buffer, shard_char_count, shard_index
    if not shard_text_buffer:
        return
    shard_filename = f'shard_{shard_index:05d}.parquet'
    for m in shard_meta_buffer:
        m['output_shard_filename'] = shard_filename
        m['output_shard_index'] = shard_index

    table = pa.Table.from_pandas(pd.DataFrame(shard_text_buffer))
    local_path = f'/tmp/{shard_filename}'
    pq.write_table(table, local_path)
    api.upload_file(
        path_or_fileobj=local_path, path_in_repo=f'{STAGE2_PREFIX}/data/{shard_filename}',
        repo_id=HF_REPO_ID, repo_type='dataset', commit_message=f'Add stage2 {shard_filename}',
    )
    os.remove(local_path)

    all_meta_records.extend(shard_meta_buffer)
    push_audit_and_manifest()

    print(f"Uploaded {shard_filename}: {len(shard_text_buffer)} docs, {shard_char_count:,} chars")
    shard_index += 1
    shard_text_buffer = []
    shard_meta_buffer = []
    shard_char_count = 0

def push_audit_and_manifest():
    audit_local = '/tmp/stage2_audit_metadata.jsonl'
    with open(audit_local, 'w') as f:
        for m in all_meta_records:
            f.write(json.dumps(m) + '\n')
    api.upload_file(
        path_or_fileobj=audit_local, path_in_repo=f'{STAGE2_PREFIX}/audit_metadata.jsonl',
        repo_id=HF_REPO_ID, repo_type='dataset', commit_message='Update stage2 audit metadata',
    )

    manifest = {
        'source_dataset': 'internet_archive',
        'stage': 'stage_2',
        'shard_config': {'chars_per_shard': STAGE2_CHARS_PER_SHARD},
        'token_budget': {
            'stage1_tokens_estimate': STAGE1_TOKENS_ESTIMATE,
            'stage2_target_tokens': STAGE2_TARGET_TOKENS,
            'stage2_hard_cap_tokens': STAGE2_HARD_CAP_TOKENS,
            'stage2_running_token_total': running_token_total,
        },
        'totals': {
            'documents': len(all_meta_records),
            'characters': sum(m['char_count'] for m in all_meta_records),
            'tokens_o200k_base': sum(m['token_count_o200k_base_gen'] for m in all_meta_records),
            'shards': shard_index + (1 if shard_text_buffer else 0),
        },
        'note': ('Cross-stage dedup against Stage 1 + final gold-digging pass deferred '
                 'until Stage 1 reprocessing (anachronistic footer / bad-token stripping) is complete.'),
    }
    manifest_local = '/tmp/stage2_manifest.json'
    with open(manifest_local, 'w') as f:
        json.dump(manifest, f, indent=2)
    api.upload_file(
        path_or_fileobj=manifest_local, path_in_repo=f'{STAGE2_PREFIX}/manifest.json',
        repo_id=HF_REPO_ID, repo_type='dataset', commit_message='Update stage2 manifest',
    )

# Priority order: math first, then broader STEM.
ordered_candidates = pd.concat([
    df_ia_candidates_dedup[df_ia_candidates_dedup['source_tag'] == 'math'],
    df_ia_candidates_dedup[df_ia_candidates_dedup['source_tag'] != 'math'],
]).to_dict('records')

# --- STEP 1: metadata-only date pre-filter (no network) --------------------
# Resolve the year from search metadata and drop anything that isn't a valid
# pre-1930 date *before* spending a download on it. Already-processed
# identifiers (resumed checkpoint) are skipped here too, so they never get
# submitted to the pool.
pending = []
for row in ordered_candidates:
    identifier = row['identifier']
    if identifier in processed:
        continue
    resolved_year = resolve_year(row)
    if resolved_year is None or resolved_year >= 1930:
        log_checkpoint(identifier, 'rejected_date')
        continue
    row['_resolved_year'] = resolved_year
    pending.append(row)

print(f"After date pre-filter: {len(pending):,} candidates to download "
      f"(of {len(ordered_candidates):,} total; the rest were already processed or failed the date check).")

# --- STEP 2: parallel download + quality gate ------------------------------
# Worker returns everything the main thread needs to decide keep/reject and to
# build a shard row. All network + CPU work happens off the main thread; the
# main thread only does ordered bookkeeping.
#
# IMPORTANT: this MUST NOT let an exception escape. fetch_ia_text already
# retries and returns None on network failure rather than raising, but we also
# wrap the whole body so that any unexpected error on a single document
# (a malformed text, an encoding edge case, etc.) is turned into a skippable
# 'rejected_error' status instead of propagating to future.result() and
# aborting the entire ~95K-item run. One bad item costs one item.
def process_candidate(row):
    identifier = row['identifier']
    try:
        text = fetch_ia_text(identifier)
        if not text:
            # fetch_ia_text returns None both for a real 404 (no djvu.txt) and
            # for exhausted retries on a flaky node; either way it's skippable.
            return identifier, 'rejected_no_text', None
        ok, result = passes_quality_bar(text)
        if not ok:
            return identifier, f'rejected_quality:{result}', None
        meta = build_metadata(row, text, row['_resolved_year'], result)
        return identifier, 'kept', (text, meta)
    except Exception:
        # Never let a single document kill the pool / the run.
        traceback.print_exc()
        return identifier, 'rejected_error', None

CHECKPOINT_PUSH_EVERY = 500
_since_last_push = 0
reached_preferred_target = running_token_total >= STAGE2_TARGET_TOKENS
stop_requested = False

if running_token_total >= STAGE2_HARD_CAP_TOKENS:
    print(f"Already at/above the Stage 2 hard cap ({running_token_total:,} >= {STAGE2_HARD_CAP_TOKENS:,} tokens) "
          f"from a prior run. Not processing any more candidates.")
    pending = []

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
    # Sliding window of in-flight futures, kept in submission (priority) order so
    # results are consumed deterministically -- identical packing to the old
    # serial loop, just with up to DOWNLOAD_WORKERS fetches overlapping.
    from collections import deque
    it = iter(pending)
    inflight = deque()

    def _submit_next():
        try:
            r = next(it)
        except StopIteration:
            return False
        inflight.append(pool.submit(process_candidate, r))
        return True

    # Prime the window.
    for _ in range(DOWNLOAD_WORKERS):
        if not _submit_next():
            break

    pbar = tqdm.tqdm(total=len(pending), desc="Stage 2 streaming extraction")
    while inflight:
        identifier, status, payload = inflight.popleft().result()  # oldest-submitted first
        pbar.update(1)

        # Keep the window full unless we've decided to stop submitting new work.
        if not stop_requested:
            _submit_next()

        if status != 'kept':
            log_checkpoint(identifier, status)
            continue

        text, meta = payload
        shard_text_buffer.append({'text': text})
        shard_meta_buffer.append(meta)
        shard_char_count += len(text)
        running_token_total += meta['token_count_o200k_base_gen']
        df_ia_gold_rows.append({**meta, 'text': text})

        log_checkpoint(identifier, 'kept')
        _since_last_push += 1

        # Token thresholds are only checked once a shard completes at its
        # natural char-based capacity -- never cut a shard short mid-way.
        if shard_char_count >= STAGE2_CHARS_PER_SHARD:
            flush_shard()
            push_checkpoint()
            _since_last_push = 0

            if running_token_total >= STAGE2_HARD_CAP_TOKENS:
                print(f"\nHard token cap reached ({running_token_total:,} >= {STAGE2_HARD_CAP_TOKENS:,} tokens) "
                      f"after completing a full shard. Draining in-flight downloads, then stopping.")
                stop_requested = True
                break
            if not reached_preferred_target and running_token_total >= STAGE2_TARGET_TOKENS:
                reached_preferred_target = True
                print(f"\nPreferred token target reached ({running_token_total:,} >= {STAGE2_TARGET_TOKENS:,} tokens) "
                      f"after completing a full shard. Draining in-flight downloads, then stopping.")
                stop_requested = True
                break
        elif _since_last_push >= CHECKPOINT_PUSH_EVERY:
            push_checkpoint()
            _since_last_push = 0

    pbar.close()
    # If we broke out early, cancel whatever is still queued (pool context exit
    # waits for the currently-running ones, which is fine -- bounded by workers).
    if stop_requested:
        for fut in inflight:
            fut.cancel()

# Final flush for any partial shard + checkpoint (covers the case where the
# candidate list was exhausted before hitting either token threshold).
flush_shard()
push_checkpoint()

df_ia_gold = pd.DataFrame(df_ia_gold_rows)
print(f"\nStage 2 run complete: {len(df_ia_gold):,} new documents this run, "
      f"{running_token_total:,} total Stage 2 tokens banked, {shard_index} shard(s) total "
      f"uploaded to {HF_REPO_ID}/{STAGE2_PREFIX}/")

### Stage 2 QA: Sampling & Statistics
Intra-Stage-2 duplicates were already removed before download (see the dedup cell above). Cross-stage dedup against Stage 1 is deferred until Stage 1's reprocessing is finalized. For now: tools to spot-check OCR quality/subject richness and view aggregate stats on what was actually extracted and uploaded to `zachnorton03/think-midtrain/stage2`.

In [ ]:
def sample_for_qa(n=5, stem_focus_tag=None, seed=None):
    """Pulls a random sample of extracted documents for manual QA of OCR
    quality and subject richness. Prints a text preview + metadata for each;
    open the identifier's IA page to compare the extracted text against the
    original scan."""
    subset = df_ia_gold if stem_focus_tag is None else df_ia_gold[df_ia_gold['stem_focus_tag'] == stem_focus_tag]
    if len(subset) == 0:
        print(f"No documents found for stem_focus_tag={stem_focus_tag!r}")
        return None
    sample = subset.sample(min(n, len(subset)), random_state=seed)
    for _, row in sample.iterrows():
        print("=" * 100)
        print(f"Title:        {row['title_src']}")
        print(f"Author:       {row['author_src']}")
        print(f"Identifier:   {row['ia_identifier']}  ({row['ia_url']})")
        print(f"Topic:        {row['topic_or_subject_gen']} (confidence {row['topic_or_subject_score_gen']})")
        print(f"Stem focus:   {row['stem_focus_tag']}")
        print(f"Year:         {row['resolved_year']}")
        print(f"OCR score (0-100, dict-hit ratio): {row['ocr_score_gen']:.2f}")
        print(f"English proportion: {row['english_proportion']:.3f}")
        print(f"Char count:   {row['char_count']:,}")
        print(f"Token count (o200k_base): {row['token_count_o200k_base_gen']:,}")
        print("-" * 100)
        print(row['text'][:1000])
        print("...")
    return sample[['ia_identifier', 'title_src', 'topic_or_subject_gen', 'stem_focus_tag', 'resolved_year', 'ocr_score_gen']]

# Example: inspect 5 random math documents for OCR quality / subject fit
sample_for_qa(n=5, stem_focus_tag='math')

In [ ]:
import matplotlib.pyplot as plt

print("--- Stage 2 (Internet Archive) Extraction Summary ---")
total_ia_chars = df_ia_gold['char_count'].sum()
print(f"Total documents:   {len(df_ia_gold):,}")
print(f"Total characters:  {total_ia_chars:,}")
print(f"Approx. GB:        {total_ia_chars / (1024**3):.2f} GB")
print(f"Total tokens (o200k_base): {df_ia_gold['token_count_o200k_base_gen'].sum():,}")
print(f"Mean OCR score:    {df_ia_gold['ocr_score_gen'].mean():.2f}")
print(f"Median OCR score:  {df_ia_gold['ocr_score_gen'].median():.2f}")
print(f"Year range:        {df_ia_gold['resolved_year'].min()} - {df_ia_gold['resolved_year'].max()}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Document count by STEM focus tier (math vs. broader STEM)
df_ia_gold['stem_focus_tag'].value_counts().plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Documents by STEM Focus Tier')
axes[0, 0].set_ylabel('Document Count')

# 2. Year distribution -- visual confirmation of the pre-1930 requirement
axes[0, 1].hist(df_ia_gold['resolved_year'], bins=30, color='darkorange', edgecolor='black')
axes[0, 1].axvline(1930, color='red', linestyle='--', label='1930 cutoff')
axes[0, 1].set_title('Publication Year Distribution')
axes[0, 1].set_xlabel('Year')
axes[0, 1].legend()

# 3. OCR score distribution (0-100 scale, matching jbduran's ocr_score_gen)
axes[1, 0].hist(df_ia_gold['ocr_score_gen'], bins=30, color='seagreen', edgecolor='black')
axes[1, 0].axvline(MIN_OCR_CONFIDENCE * 100, color='red', linestyle='--', label=f'gate ({MIN_OCR_CONFIDENCE * 100:.0f})')
axes[1, 0].set_title('OCR Score Distribution')
axes[1, 0].set_xlabel('OCR Score (0-100, dict-hit ratio)')
axes[1, 0].legend()

# 4. Top LC-style topic buckets -- richness check
top_subjects = df_ia_gold['topic_or_subject_gen'].value_counts().head(15)
top_subjects.plot(kind='barh', ax=axes[1, 1], color='mediumpurple')
axes[1, 1].set_title('Documents by Topic Bucket (richness check)')
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()

### Final Validation
This section ensures the generated shards are ready for training.

In [ ]:
# Verification of shard integrity post-upload -- reads back from HF rather
# than local disk, since local disk is ephemeral and HF is the source of truth.
manifest_check = hf_hub_download(repo_id=HF_REPO_ID, filename=f'{STAGE2_PREFIX}/manifest.json', repo_type='dataset', token=HF_TOKEN)
with open(manifest_check) as f:
    print(json.dumps(json.load(f), indent=2))

first_shard_check = hf_hub_download(repo_id=HF_REPO_ID, filename=f'{STAGE2_PREFIX}/data/shard_00000.parquet', repo_type='dataset', token=HF_TOKEN)
display(pq.read_table(first_shard_check).to_pandas().head())